[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/PacktPublishing/Modern-Computer-Vision-with-PyTorch-2E/blob/main/Chapter10/action_recognition.ipynb)

In [29]:
%%capture
# This will take approx 3 min
%pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu121
%pip install -U "openmim==0.3.9"
!mim install -U "mmengine==0.10.4"
!mim install "mmcv==2.2.0"
!git clone https://github.com/sizhky/mmaction2.git -b main
%cd mmaction2
%pip install -e .
%pip install -r requirements/optional.txt
%pip install "timm==0.9.16"
%pip install "torch-snippets==0.528" lovely-tensors

In [30]:
# Restore a compatible modern setuptools
!pip install -U "setuptools>=70.0.0"
# Install mmcv-lite directly to bypass C++/CUDA compilation on Python 3.13
!pip install "mmcv-lite>=2.0.0rc4"

In [31]:
# Check Pytorch installation
import torch, torchvision
print("PyTorch version:", torch.__version__, "| CUDA available:", torch.cuda.is_available())

# Check MMAction2 installation
try:
    import mmaction
    print("MMAction2 version:", mmaction.__version__)
except ImportError as e:
    print("MMAction2 Import failed:", e)

# Check MMCV installation and state
try:
    import mmcv
    print("MMCV wrapper imported successfully.")
    from mmcv.ops import get_compiling_cuda_version, get_compiler_version
    print("CUDA compiler version:", get_compiling_cuda_version())
except ImportError as e:
    print("\n[INFO] Full MMCV (with CUDA extensions) is missing or incomplete:", e)
    print("[ACTION] mmcv-lite was used. To resolve this, we need to compile the full 'mmcv' package from source for Python 3.13.")

PyTorch version: 2.14.1+cu130 | CUDA available: True
MMAction2 version: 1.2.0
MMCV wrapper imported successfully.

[INFO] Full MMCV (with CUDA extensions) is missing or incomplete: No module named 'mmcv._ext'
[ACTION] mmcv-lite was used. To resolve this, we need to compile the full 'mmcv' package from source for Python 3.13.


### Download TSN Model Checkpoints
We will download the pre-trained weights for the `tsn_r50_1x1x3` model on the Kinetics-400 dataset.

In [32]:
# Create checkpoints directory and download pre-trained weights
!mkdir -p checkpoints
!wget -q -c https://download.openmmlab.com/mmaction/recognition/tsn/tsn_r50_1x1x3_100e_kinetics400_rgb/tsn_r50_1x1x3_100e_kinetics400_rgb_20200614-e508be42.pth \
      -O checkpoints/tsn_r50_1x1x3_100e_kinetics400_rgb_20200614-e508be42.pth
print("Checkpoint download complete.")

Checkpoint download complete.


### Initialize TSN Recognizer
We load the configuration and feed the downloaded checkpoint into the `init_recognizer` API.

In [33]:
import sys
from types import ModuleType

# Create a mock module structure for multimodal to block any downstream loads
mock_multimodal = ModuleType('mmaction.models.multimodal')
mock_multimodal.WITH_MULTIMODAL = False
sys.modules['mmaction.models.multimodal'] = mock_multimodal

# Force the dependency utility flags to False before importing anything else
try:
    import mmaction.utils.dependency as dep_utils
    dep_utils.WITH_MULTIMODAL = False
except ImportError:
    pass

# Now import the API and engine safely
from mmaction.apis import init_recognizer
from mmengine import Config
import torch

# Path to the config file
config_path = 'configs/recognition/tsn/tsn_imagenet-pretrained-r50_8xb32-1x1x3-100e_kinetics400-rgb.py'
cfg = Config.fromfile(config_path)

# Setup checkpoint path
checkpoint_path = 'checkpoints/tsn_r50_1x1x3_100e_kinetics400_rgb_20200614-e508be42.pth'

# Initialize model on GPU (cuda:0) if available, otherwise CPU
device = 'cuda:0' if torch.cuda.is_available() else 'cpu'
model = init_recognizer(cfg, checkpoint_path, device=device)
print(f"Model initialized successfully on device: {device}")

Loads checkpoint by local backend from path: checkpoints/tsn_r50_1x1x3_100e_kinetics400_rgb_20200614-e508be42.pth
Model initialized successfully on device: cuda:0


## Perform inference with a MMAction2 recognizer
MMAction2 already provides high level APIs to do inference and training.

In [34]:
!mkdir checkpoints
!wget -c https://download.openmmlab.com/mmaction/recognition/tsn/tsn_r50_1x1x3_100e_kinetics400_rgb/tsn_r50_1x1x3_100e_kinetics400_rgb_20200614-e508be42.pth \
      -O checkpoints/tsn_r50_1x1x3_100e_kinetics400_rgb_20200614-e508be42.pth

mkdir: cannot create directory ‘checkpoints’: File exists
--2026-11-10 03:28:10--  https://download.openmmlab.com/mmaction/recognition/tsn/tsn_r50_1x1x3_100e_kinetics400_rgb/tsn_r50_1x1x3_100e_kinetics400_rgb_20200614-e508be42.pth
Resolving download.openmmlab.com (download.openmmlab.com)... 163.181.246.192, 163.181.246.193, 163.181.246.194, ...
Connecting to download.openmmlab.com (download.openmmlab.com)|163.181.246.192|:443... connected.
HTTP request sent, awaiting response... 416 Requested Range Not Satisfiable

    The file is already fully retrieved; nothing to do.



In [35]:
# Install decord for video frame decoding
!pip install -q decord

# Run inference directly using MMAction2 APIs
from mmaction.apis import inference_recognizer

# We will use the already initialized 'model' variable from the previous cell
video = 'demo/demo.mp4'

print(f"Running action recognition inference on video: {video}...")
results = inference_recognizer(model, video)

# Extract raw predicted scores
pred_scores = results.pred_score.tolist()
print("Inference completed successfully!")

Running action recognition inference on video: demo/demo.mp4...
Inference completed successfully!


In [36]:
model

Recognizer2D(
  (data_preprocessor): ActionDataPreprocessor()
  (backbone): ResNet(
    (conv1): ConvModule(
      (conv): Conv2d(3, 64, kernel_size=(7, 7), stride=(2, 2), padding=(3, 3), bias=False)
      (bn): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
      (activate): ReLU(inplace=True)
    )
    (maxpool): MaxPool2d(kernel_size=3, stride=2, padding=1, dilation=1, ceil_mode=False)
    (layer1): Sequential(
      (0): Bottleneck(
        (conv1): ConvModule(
          (conv): Conv2d(64, 64, kernel_size=(1, 1), stride=(1, 1), bias=False)
          (bn): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
          (activate): ReLU(inplace=True)
        )
        (conv2): ConvModule(
          (conv): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
          (bn): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
          (activ

In [37]:
# Use the recognizer to format and output predicted action labels
from operator import itemgetter

def predict():
    video = 'demo/demo.mp4'
    label = 'tools/data/kinetics/label_map_k400.txt'
    results = inference_recognizer(model, video)

    pred_scores = results.pred_score.tolist()
    score_tuples = tuple(zip(range(len(pred_scores)), pred_scores))
    score_sorted = sorted(score_tuples, key=itemgetter(1), reverse=True)
    top5_label = score_sorted[:5]

    labels = open(label).readlines()
    labels = [x.strip() for x in labels]
    results = [(labels[k[0]], k[1]) for k in top5_label]
    return results

# Run the prediction and store results directly without torch_snippets wrappers
results = predict()

In [38]:
print('The top-5 labels with corresponding scores are:')
for result in results:
    print(f'{result[0]}: ', result[1])

The top-5 labels with corresponding scores are:
arm wrestling:  1.0
rock scissors paper:  6.434465404936418e-09
shaking hands:  2.759991346579227e-09
clapping:  1.345463851443185e-09
massaging feet:  5.555121918021655e-10


## Train a recognizer on customized dataset

To train a new recognizer, there are usually three things to do:
1. Support a new dataset
2. Modify the config
3. Train a new recognizer

### Support a new dataset

In this tutorial, we gives an example to convert the data into the format of existing datasets. Other methods and more advanced usages can be found in the [doc](/docs/tutorials/new_dataset.md)

Firstly, let's download a tiny dataset obtained from [Kinetics-400](https://deepmind.com/research/open-source/open-source-datasets/kinetics/). We select 30 videos with their labels as train dataset and 10 videos with their labels as test dataset.

In [39]:
# download, decompress the data
!rm kinetics400_tiny.zip*
!rm -rf kinetics400_tiny
!wget https://download.openmmlab.com/mmaction/kinetics400_tiny.zip
!unzip kinetics400_tiny.zip > /dev/null

rm: cannot remove 'kinetics400_tiny.zip*': No such file or directory
--2026-11-10 03:28:21--  https://download.openmmlab.com/mmaction/kinetics400_tiny.zip
Resolving download.openmmlab.com (download.openmmlab.com)... 163.181.246.191, 163.181.246.192, 163.181.246.193, ...
Connecting to download.openmmlab.com (download.openmmlab.com)|163.181.246.191|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 18308682 (17M) [application/zip]
Saving to: ‘kinetics400_tiny.zip’

kinetics400_tiny.zi 100%[===================>]  17.46M  4.89MB/s    in 3.9s    

2026-11-10 03:28:25 (4.44 MB/s) - ‘kinetics400_tiny.zip’ saved [18308682/18308682]



In [40]:
from glob import glob
glob('kinetics400_tiny/*/*')

['kinetics400_tiny/train/D32_1gwq35E.mp4',
 'kinetics400_tiny/train/phDqGd0NKoo.mp4',
 'kinetics400_tiny/train/xGY2dP0YUjA.mp4',
 'kinetics400_tiny/train/TkkZPZHbAKA.mp4',
 'kinetics400_tiny/train/T_TMNGzVrDk.mp4',
 'kinetics400_tiny/train/IyfILH9lBRo.mp4',
 'kinetics400_tiny/train/D92m0HsHjcQ.mp4',
 'kinetics400_tiny/train/yLC9CtWU5ws.mp4',
 'kinetics400_tiny/train/34XczvTaRiI.mp4',
 'kinetics400_tiny/train/WWP5HZJsg-o.mp4',
 'kinetics400_tiny/train/DbX8mPslRXg.mp4',
 'kinetics400_tiny/train/h2YqqUhnR34.mp4',
 'kinetics400_tiny/train/WaS0qwP46Us.mp4',
 'kinetics400_tiny/train/RqnKtCEoEcA.mp4',
 'kinetics400_tiny/train/oMrZaozOvdQ.mp4',
 'kinetics400_tiny/train/soEcZZsBmDs.mp4',
 'kinetics400_tiny/train/P5M-hAts7MQ.mp4',
 'kinetics400_tiny/train/oXy-e_P_cAI.mp4',
 'kinetics400_tiny/train/27_CSXByd3s.mp4',
 'kinetics400_tiny/train/O46YA8tI530.mp4',
 'kinetics400_tiny/train/LvcFDgCAXQs.mp4',
 'kinetics400_tiny/train/ZQV4U2KQ370.mp4',
 'kinetics400_tiny/train/PnOe3GZRVX8.mp4',
 'kinetics4

In [42]:
# After downloading the data, we need to check the annotation format
!cat kinetics400_tiny/kinetics_tiny_train_video.txt

D32_1gwq35E.mp4 0
iRuyZSKhHRg.mp4 1
oXy-e_P_cAI.mp4 0
34XczvTaRiI.mp4 1
h2YqqUhnR34.mp4 0
O46YA8tI530.mp4 0
kFC3KY2bOP8.mp4 1
WWP5HZJsg-o.mp4 1
phDqGd0NKoo.mp4 1
yLC9CtWU5ws.mp4 0
27_CSXByd3s.mp4 1
IyfILH9lBRo.mp4 1
T_TMNGzVrDk.mp4 1
TkkZPZHbAKA.mp4 0
PnOe3GZRVX8.mp4 1
soEcZZsBmDs.mp4 1
FMlSTTpN3VY.mp4 1
WaS0qwP46Us.mp4 0
A-wiliK50Zw.mp4 1
oMrZaozOvdQ.mp4 1
ZQV4U2KQ370.mp4 0
DbX8mPslRXg.mp4 1
h10B9SVE-nk.mp4 1
P5M-hAts7MQ.mp4 0
R8HXQkdgKWA.mp4 0
D92m0HsHjcQ.mp4 0
RqnKtCEoEcA.mp4 0
LvcFDgCAXQs.mp4 0
xGY2dP0YUjA.mp4 0
Wh_YPQdH1Zg.mp4 0


According to the format defined in [`VideoDataset`](./datasets/video_dataset.py), each line indicates a sample video with the filepath and label, which are split with a whitespace.

### Modify the config

In the next step, we need to modify the config for the training.
To accelerate the process, we finetune a recognizer using a pre-trained recognizer.

In [43]:
from mmengine import Config
cfg = Config.fromfile('./configs/recognition/tsn/tsn_imagenet-pretrained-r50_8xb32-1x1x3-100e_kinetics400-rgb.py')

Given a config that trains a TSN model on kinetics400-full dataset, we need to modify some values to use it for training TSN on Kinetics400-tiny dataset.


In [44]:
from mmengine.runner import set_random_seed

# Modify dataset type and path
cfg.data_root = 'kinetics400_tiny/train/'
cfg.data_root_val = 'kinetics400_tiny/val/'
cfg.ann_file_train = 'kinetics400_tiny/kinetics_tiny_train_video.txt'
cfg.ann_file_val = 'kinetics400_tiny/kinetics_tiny_val_video.txt'


cfg.test_dataloader.dataset.ann_file = 'kinetics400_tiny/kinetics_tiny_val_video.txt'
cfg.test_dataloader.dataset.data_prefix.video = 'kinetics400_tiny/val/'

cfg.train_dataloader.dataset.ann_file = 'kinetics400_tiny/kinetics_tiny_train_video.txt'
cfg.train_dataloader.dataset.data_prefix.video = 'kinetics400_tiny/train/'

cfg.val_dataloader.dataset.ann_file = 'kinetics400_tiny/kinetics_tiny_val_video.txt'
cfg.val_dataloader.dataset.data_prefix.video  = 'kinetics400_tiny/val/'


# Modify num classes of the model in cls_head
cfg.model.cls_head.num_classes = 2
# We can use the pre-trained TSN model
cfg.load_from = './checkpoints/tsn_r50_1x1x3_100e_kinetics400_rgb_20200614-e508be42.pth'

# Set up working dir to save files and logs.
cfg.work_dir = './output'

# The original learning rate (LR) is set for 8-GPU training.
# We divide it by 8 since we only use one GPU.
cfg.train_dataloader.batch_size = cfg.train_dataloader.batch_size // 16
cfg.val_dataloader.batch_size = cfg.val_dataloader.batch_size // 16
cfg.optim_wrapper.optimizer.lr = cfg.optim_wrapper.optimizer.lr / 8 / 16
cfg.train_cfg.max_epochs = 10

cfg.train_dataloader.num_workers = 2
cfg.val_dataloader.num_workers = 2
cfg.test_dataloader.num_workers = 2

# We can initialize the logger for training and have a look
# at the final config used for training
print(f'Config:\n{cfg.pretty_text}')


Config:
ann_file_train = 'kinetics400_tiny/kinetics_tiny_train_video.txt'
ann_file_val = 'kinetics400_tiny/kinetics_tiny_val_video.txt'
auto_scale_lr = dict(base_batch_size=256, enable=False)
data_root = 'kinetics400_tiny/train/'
data_root_val = 'kinetics400_tiny/val/'
dataset_type = 'VideoDataset'
default_hooks = dict(
    checkpoint=dict(
        interval=3, max_keep_ckpts=3, save_best='auto', type='CheckpointHook'),
    logger=dict(ignore_last=False, interval=20, type='LoggerHook'),
    param_scheduler=dict(type='ParamSchedulerHook'),
    runtime_info=dict(type='RuntimeInfoHook'),
    sampler_seed=dict(type='DistSamplerSeedHook'),
    sync_buffers=dict(type='SyncBuffersHook'),
    timer=dict(type='IterTimerHook'))
default_scope = 'mmaction'
env_cfg = dict(
    cudnn_benchmark=False,
    dist_cfg=dict(backend='nccl'),
    mp_cfg=dict(mp_start_method='fork', opencv_num_threads=0))
file_client_args = dict(io_backend='disk')
load_from = './checkpoints/tsn_r50_1x1x3_100e_kinetics400_rgb_

### Train a new recognizer

Finally, lets initialize the dataset and recognizer, then train a new recognizer!

In [ ]:
import os.path as osp
import mmengine
from mmengine.runner import Runner

# Create work_dir
mmengine.mkdir_or_exist(osp.abspath(cfg.work_dir))

# build the runner from config
runner = Runner.from_cfg(cfg)

# start training
trained_model = runner.train()

11/10 03:28:27 - mmengine - INFO - 
------------------------------------------------------------
System environment:
    sys.platform: linux
    Python: 3.13.16 (main, Oct  1 2026, 15:40:24) [GCC 13.3.0]
    CUDA available: True
    MUSA available: False
    numpy_random_seed: 957271774
    GPU 0: Tesla T4
    CUDA_HOME: /usr/local/cuda
    NVCC: Cuda compilation tools, release 13.0, V13.0.88
    GCC: x86_64-linux-gnu-gcc (Ubuntu 13.3.0-6ubuntu2~24.04.1) 13.3.0
    PyTorch: 2.14.1+cu130
    PyTorch compiling details: PyTorch built with:
  - GCC 13.3
  - C++ Version: 202002
  - Intel(R) oneAPI Math Kernel Library Version 2024.2-Product Build 20240605 for Intel(R) 64 architecture applications
  - Intel(R) MKL-DNN v3.12.0 (Git Hash 80afa71049cd69a3df32adcccb623b12cd7baa22)
  - OpenMP 201511 (a.k.a. OpenMP 4.5)
  - LAPACK is enabled (usually provided by MKL)
  - NNPACK is enabled
  - CPU capability usage: AVX512
  - CUDA Runtime 13.0
  - NVCC architecture flags: -gencode;arch=compute_75,co

In [ ]:
runner.test()